# Lorenz Water Wheel — TCN training (Colab)

Train the PyTorch Temporal Convolutional Network (`src/tcn/tcn_forecast.py`) on
Colab's GPU, then save the checkpoint into the project's `weights/` directory and
commit it so `notebooks/evaluate_colab.ipynb` can score it.

**Before running:** set the runtime to GPU via *Runtime → Change runtime type → GPU*.

## 1. Get the code

In [ ]:
# Clone the repo. BRANCH selects what to pull: "main" for the merged code, or a feature
# branch name to test changes before they are merged. --depth 1 grabs only the latest
# snapshot (faster); rm -rf makes re-running safe (a plain re-clone would silently keep
# the old dir).
BRANCH = "main"
%cd /content
!rm -rf Lorenz-wheel-prediction-LSTM
!git clone --depth 1 -b {BRANCH} https://github.com/antmatrik/Lorenz-wheel-prediction-LSTM.git
%cd Lorenz-wheel-prediction-LSTM

## 2. Install dependencies (TCN only)

In [ ]:
!pip install -q -r requirements/tcn.txt

## 3. Train

Run **either** the sanity cell **or** the production cell. Both write
`outputs/tcn_checkpoint.pt` (weights + scaler + window + architecture in one file).

### 3a. Sanity run — 1 epoch on 1 file

In [ ]:
!python src/tcn/tcn_forecast.py --mode train --sanity

### 3b. Production run

Tuned defaults, adapted to our CLI. `--patience` early-stops once val loss plateaus (so
`--epochs` is just a cap), the best checkpoint is re-saved every time val improves, and the
current learning rate is logged each epoch (`ReduceLROnPlateau` shrinks it on plateaus).
More `--channels` layers widen the receptive field (use more of `--window`) but train
slower. See all options with `!python src/tcn/tcn_forecast.py --help`.

In [ ]:
# --epochs is a cap; --patience 8 stops early once val loss plateaus.
# Receptive-field vs speed: 4 layers ~60 steps; 6 layers ~250 (covers a 256 window).
# To widen it, add layers, e.g. --channels 128,128,128,128,128,128
!python src/tcn/tcn_forecast.py --mode train \
  --window 256 \
  --channels 128,128,128,128 \
  --learning-rate 2e-4 \
  --batch-size 256 \
  --epochs 60 \
  --patience 8

## 4. Quick evaluation (optional)

Score the freshly trained TCN on `data/test-dataset` (native 3-channel rollout;
add `--physics` for the LSTM-style rollout).

In [ ]:
!python src/common/evaluate.py --model tcn
!python src/common/evaluate.py --model tcn --horizons 25,50,100,200,400,800,1800

## 5. Save the checkpoint into the project

Copy the checkpoint to `weights/`, then **download it and commit it to the repo**
(`git add weights/ && git commit && git push`) so the evaluation notebook can use it.

In [ ]:
import os, shutil
os.makedirs('weights', exist_ok=True)
src = 'outputs/tcn_checkpoint.pt'
if os.path.exists(src):
    shutil.copy(src, 'weights/tcn_checkpoint.pt'); print('saved weights/tcn_checkpoint.pt')
else:
    print('skip (not found):', src)

### Download to your machine

In [ ]:
from google.colab import files
files.download('weights/tcn_checkpoint.pt')

### Alternative: copy to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
DRIVE_DIR = '/content/drive/MyDrive/lorenz_waterwheel'
os.makedirs(DRIVE_DIR, exist_ok=True)
src = 'weights/tcn_checkpoint.pt'
if os.path.exists(src):
    shutil.copy(src, os.path.join(DRIVE_DIR, 'tcn_checkpoint.pt')); print('saved tcn_checkpoint.pt')